In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import entropy, skew
import statsmodels.api as sm

In [ ]:
# LOAD DATA
spy = pd.read_csv("spy_1min_data")
news = pd.read_csv("news_data.csv")
daily = pd.read_csv("news_daily_data")

spy["date"] = pd.to_datetime(spy["date"])
news["date"] = pd.to_datetime(news["date"])
daily["date"] = pd.to_datetime(daily["date"])

/var/folders/tw/hrpgvqqj41q77z58rc7zg9n00000gn/T/ipykernel_1115/696058300.py:3: DtypeWarning: Columns (0: section_name) have mixed types. Specify dtype option on import or set low_memory=False.
  news = pd.read_csv("news_data.csv")


In [ ]:
news["date"] = news["date"].dt.tz_convert(None)
news["date"] = news["date"].dt.strftime("%Y-%m-%d %H:%M:%S")
news["date"] = pd.to_datetime(news["date"])

In [ ]:
# SAFE ENCODING

def safe_encode(s):
    mapping = {k: i for i, k in enumerate(sorted(s.dropna().astype(str).unique()))}
    return s.astype(str).map(mapping)

for col in [
    "sentiment_class",
    "surprise_class",
    "day_uncertainty_class",
    "day_disagreement_class"
]:
    news[col + "_enc"] = safe_encode(news[col])

In [ ]:
# NORMALIZATION

def norm(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

for col in [
    "sentiment_class_enc",
    "surprise_class_enc",
    "day_uncertainty_class_enc",
    "day_disagreement_class_enc"
]:
    news[col] = norm(news[col])

In [ ]:
# CORE FEATURES (NEWS SIDE)

def disagreement(x):
    return np.std(x)

def sentiment_entropy(x):

    hist, _ = np.histogram(
        x,
        bins=min(5, len(np.unique(x))),
        density=True
    )

    hist = hist[hist > 0]

    return entropy(hist)

def sentiment_skew(x):

    if len(np.unique(x)) < 2:
        return 0

    return skew(x)

# def entropy_feat(x):
#     hist, _ = np.histogram(x, bins=5, density=True)
#     hist = hist + 1e-8
#     return entropy(hist)

In [ ]:
spy["return"] = spy["close"].pct_change()
spy["abs_return"] = spy["return"].abs()

spy["rv_5m"] = spy["return"].rolling(5).std()
spy["rv_15m"] = spy["return"].rolling(15).std()
spy["rv_60m"] = spy["return"].rolling(60).std()

spy["volume_z"] = (spy["volume"] - spy["volume"].mean()) / spy["volume"].std()

In [ ]:
# MARKET REACTION FUNCTION

def get_market(t):
    out = {}

    for w in [5, 15, 60]:
        window = spy[
            (spy["date"] >= t) &
            (spy["date"] <= t + pd.Timedelta(minutes=w))
        ]

        if len(window) < 3:
            continue

        r = window["return"].dropna()

        out[f"rv_{w}m"] = r.std()
        out[f"max_move_{w}m"] = r.abs().max()
        out[f"volume_spike_{w}m"] = window["volume_z"].mean()

    return out

In [ ]:
# BUILD EVENT DATASET

news["event_time"] = (
    news["date"]
    .dt.floor("10min")
)

data=[]

for t, window_news in news.groupby("event_time"):

    if len(window_news)<2:
        continue

    sent = window_news["sentiment_class_enc"].values

    record = {

        "timestamp": t,

        "n_news": len(window_news),

        "mean_sentiment":
            sent.mean(),

        "event_disagreement":
            sent.std(),

        "sent_entropy":
            sentiment_entropy(sent),

        "skew_sentiment":
            sentiment_skew(sent),

        "day_disagreement":
            window_news[
                "day_disagreement_class_enc"
            ].iloc[0],

        "uncertainty":
            window_news[
                "day_uncertainty_class_enc"
            ].iloc[0],

        "surprise":
            window_news[
                "surprise_class_enc"
            ].mean()
    }

    market=get_market(t)

    if market:
        data.append({
            **record,
            **market
        })

df=pd.DataFrame(data)

In [ ]:
# CORE QUESTION 1 — DISAGREEMENT → VOLATILITY

df.groupby(
    pd.qcut(
        df["event_disagreement"],
        3,
        duplicates="drop"
    )
)[
    [
        "rv_5m",
        "rv_15m",
        "rv_60m"
    ]
].mean()

,rv_5m,rv_15m,rv_60m
event_disagreement,,,
"(-0.001, 0.125]",0.000382,0.000400,0.000447
"(0.125, 0.408]",0.000413,0.000426,0.000501


In [ ]:
# CORE QUESTION 2 — ENTROPY → TAIL RISK


# Top 10% volatility events
df["tail"] = (
    df["rv_60m"]
    >
    df["rv_60m"].quantile(0.90)
).astype(int)

df.groupby(
    pd.qcut(
        df["sent_entropy"],
        3,
        duplicates="drop"
    )
)["tail"].mean()

sent_entropy
(-0.001, 0.673]    0.096071
(0.673, 1.609]     0.108131
Name: tail, dtype: float64

In [ ]:
# CORE QUESTION 3 — VOLUME EFFECT

df.groupby(
    pd.qcut(
        df["event_disagreement"],
        3,
        duplicates="drop"
    )
)[
    [
        "volume_spike_5m",
        "volume_spike_15m",
        "volume_spike_60m"
    ]
].mean()

,volume_spike_5m,volume_spike_15m,volume_spike_60m
event_disagreement,,,
"(-0.001, 0.125]",-0.002856,0.008042,0.040388
"(0.125, 0.408]",0.009087,0.020524,0.046014


In [ ]:
# Correlation Check

corr = df[
    [
        "event_disagreement",
        "sent_entropy",
        "skew_sentiment",
        "uncertainty",
        "surprise"
    ]
].corr()

print(corr)

                    event_disagreement  sent_entropy  skew_sentiment  \
event_disagreement            1.000000      0.925051       -0.041665   
sent_entropy                  0.925051      1.000000       -0.048177   
skew_sentiment               -0.041665     -0.048177        1.000000   
uncertainty                   0.071693      0.062711       -0.045952   
surprise                      0.418552      0.380555        0.037719   

                    uncertainty  surprise  
event_disagreement     0.071693  0.418552  
sent_entropy           0.062711  0.380555  
skew_sentiment        -0.045952  0.037719  
uncertainty            1.000000  0.025316  
surprise               0.025316  1.000000  


In [ ]:
# VIF Check

from statsmodels.stats.outliers_influence import variance_inflation_factor
X_vif = df[
    [
        "event_disagreement",
        "sent_entropy",
        "skew_sentiment",
        "uncertainty",
        "surprise"
    ]
].copy()

In [ ]:
vif = pd.DataFrame()

vif["feature"] = X_vif.columns

vif["VIF"] = [

    variance_inflation_factor(
        X_vif.values,
        i
    )

    for i in range(len(X_vif.columns))
]

print(vif)

              feature        VIF
0  event_disagreement  12.898665
1        sent_entropy  13.117338
2      skew_sentiment   1.016112
3         uncertainty   2.014024
4            surprise   2.024312


In [ ]:
# MODEL 1
## Disagreement → Volatility

X = df[
    [
        "event_disagreement",
        # "sent_entropy",
        "skew_sentiment",
        "uncertainty",
        "surprise"
    ]
]

X = sm.add_constant(X)

y = df["rv_60m"]

model_vol = sm.OLS(
    y,
    X
).fit(cov_type="HC3")

print(model_vol.summary())

                            OLS Regression Results                            
Dep. Variable:                 rv_60m   R-squared:                       0.037
Model:                            OLS   Adj. R-squared:                  0.037
Method:                 Least Squares   F-statistic:                     91.29
Date:                Thu, 11 Jun 2026   Prob (F-statistic):           1.89e-76
Time:                        15:46:53   Log-Likelihood:                 68017.
No. Observations:               10745   AIC:                        -1.360e+05
Df Residuals:                   10740   BIC:                        -1.360e+05
Df Model:                           4                                         
Covariance Type:                  HC3                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                  0.0002   1.22

In [ ]:
df["event_id"] = np.arange(len(df))

df.nlargest(20, "skew_sentiment")[
    [
        "event_id",
        "skew_sentiment",
        "event_disagreement",
        "rv_60m",
        "volume_spike_60m"
    ]
]

,event_id,skew_sentiment,event_disagreement,rv_60m,volume_spike_60m
6877,6877,2.846050,0.071870,0.001622,0.193685
10270,10270,2.666667,0.075000,0.000308,-0.147621
2179,2179,2.267787,0.082680,0.000450,0.051418
7247,7247,2.267787,0.082680,0.000598,0.058013
8502,8502,2.267787,0.082680,0.000240,-0.101560
1970,1970,2.041241,0.087482,0.000629,0.195394
6968,6968,2.041241,0.087482,0.000460,0.108291
10341,10341,2.041241,0.087482,0.000285,-0.100745
956,956,2.041241,0.087482,0.000778,0.166975
2095,2095,2.041241,0.087482,0.000836,0.076516


In [ ]:
df.nsmallest(20, "skew_sentiment")[
    [
        "event_id",
        "skew_sentiment",
        "event_disagreement",
        "rv_60m",
        "volume_spike_60m"
    ]
]

,event_id,skew_sentiment,event_disagreement,rv_60m,volume_spike_60m
351,351,-2.666667,0.075000,0.001215,-0.117927
2769,2769,-2.474874,0.078567,0.000986,0.008000
7953,7953,-2.474874,0.078567,0.000849,0.130980
8517,8517,-2.474874,0.078567,0.000293,-0.086885
9199,9199,-2.474874,0.078567,0.001009,0.065007
6576,6576,-2.267787,0.082680,0.001439,0.240200
6920,6920,-2.267787,0.082680,0.000617,0.035574
7345,7345,-2.267787,0.082680,0.000596,0.076098
9027,9027,-2.267787,0.082680,0.000734,-0.002799
10144,10144,-2.267787,0.082680,0.000598,-0.040858


In [ ]:
extreme = df.nlargest(5, "skew_sentiment")

for t in extreme["timestamp"]:
    print("="*50)
    print(t)

    display(
        news[
            (news["date"] >= t - pd.Timedelta(minutes=10))
            &
            (news["date"] <= t + pd.Timedelta(minutes=10))
        ][["headline","sentiment_class"]]
    )

2023-05-05 09:00:00


,headline,sentiment_class
31552,Inside King Charles III’s Coronation Coaches,C
31553,The Bearer of Bad News,C
31554,Wonks ‘Socialize’ Differently,C
31555,U.S. Job Growth Retains Vigor Despite Economic...,D
31556,‘Ron DeSoros’? Conspiracy Theorists Target Tru...,C
31557,Businesses Caught in Cross-Fire as Iran Enforc...,C
31558,"As a King Is Crowned, Some Britons Ask Why the...",C
31559,He’s 86 and Long Retired. Why Are Israelis Pro...,C
31560,Indian Olympians Persist in Demanding Arrest o...,C
109017,Keir Starmer: Labour on track to win general e...,C


2025-01-29 10:00:00


,headline,sentiment_class
48894,"He Survived 15 Months of War in Gaza, Then Die...",C
48895,"The Fed Holds Rates Steady, Hitting Pause Afte...",C
48896,Sundance Made Park City the It Town. Not Anymore.,C
48897,They Invested Billions. Then the A.I. Script G...,C
48898,Meta Engineers See Vindication in DeepSeek’s A...,D
48899,Trump Is Eyeing Greenland. His Commerce Nomine...,C
48900,Citizenship by Birthright? By Bloodline? Migra...,C
138609,UK reportedly planning electric car loan subsi...,C
138610,‘Africa is where I’m from’: why some Black Bra...,C
138611,‘Headed for technofascism’: the rightwing root...,C


2021-03-31 09:00:00


,headline,sentiment_class
9486,See How Rich Countries Got to the Front of the...,C
9487,"If You Care About Privacy, It’s Time to Try a ...",C
9488,Turing Award Goes to Creators of Computer Prog...,C
9489,"Building a Mosque in France, Never Easy, May G...",C
9490,Biden Details $2 Trillion Plan to Rebuild Infr...,D
9491,Britain’s undocumented immigrants are entitled...,C
70321,Downing Street suggests UK should be seen as m...,C
70322,"The father, the son and the racist spirit: bei...",C


2023-07-07 09:00:00


,headline,sentiment_class
33141,The ‘Whisper Listing’ Gets Louder,C
33142,Gen X Is in Charge. Don’t Make a Big Deal Abou...,C
33143,"Donors Are War-Weary, So Ukrainian Soldiers Ge...",C
33144,U.S. Raises Pressure on China to Combat Global...,C
33145,Millions Danced Joyfully to Her Song. She Drew...,C
33146,Fed Watchers to Parse June Jobs Report Closely,C
33147,"Against the Odds, the U.S. Economy Chugs Along...",D
33148,How Do ‘Barbie’ and Blackpink Figure in a Dang...,C


2024-02-21 10:00:00


,headline,sentiment_class
39322,"In Latin America, Guards Don’t Control Prisons...",C
39323,"Are We in a Productivity Boom? For Clues, Look...",D
39324,Israeli Raid in West Bank City of Jenin Kills ...,C
39325,How Productivity Affects the Economy,C
39326,Russian Forces Press On With Attacks in Southe...,C
39327,A Missing Scottish Trophy Will Be Awarded Agai...,C
123036,"Emotional, messy and breathtakingly ruthless: ...",C
123037,Keir Starmer was beaten up as teenager trying ...,C


In [ ]:
# MODEL 2
## Disagreement → Volume

y = df["volume_spike_60m"]

model_volume = sm.OLS(
    y,
    X
).fit(cov_type="HC3")

print(model_volume.summary())

                            OLS Regression Results                            
Dep. Variable:       volume_spike_60m   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     4.113
Date:                Thu, 11 Jun 2026   Prob (F-statistic):            0.00248
Time:                        15:46:58   Log-Likelihood:                 152.53
No. Observations:               10745   AIC:                            -295.1
Df Residuals:                   10740   BIC:                            -258.6
Df Model:                           4                                         
Covariance Type:                  HC3                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                  0.0146      0

In [ ]:
# # MODEL 3
# ## Tail Risk

# df["tail"] = (
#     df["rv_60m"]
#     >
#     df["rv_60m"].quantile(0.90)
# ).astype(int)

In [ ]:
import statsmodels.api as sm

X = df[
    [
        "event_disagreement",
        # "sent_entropy",
        "skew_sentiment",
        "uncertainty",
        "surprise"
    ]
]

X = sm.add_constant(X)

y = df["tail"]

In [ ]:
tail_model = sm.Logit(
    y,
    X
).fit()

print(tail_model.summary())

Optimization terminated successfully.
         Current function value: 0.314925
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                   tail   No. Observations:                10745
Model:                          Logit   Df Residuals:                    10740
Method:                           MLE   Df Model:                            4
Date:                Thu, 11 Jun 2026   Pseudo R-squ.:                 0.03155
Time:                        15:47:00   Log-Likelihood:                -3383.9
converged:                       True   LL-Null:                       -3494.1
Covariance Type:            nonrobust   LLR p-value:                 1.471e-46
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -3.6419      0.122    -29.869      0.000      -3.881      -3.403
event

In [ ]:
df["abs_skew"] = np.abs(
    df["skew_sentiment"]
)

df.groupby(
    pd.qcut(
        df["abs_skew"],
        5,
        duplicates="drop"
    )
)[
    ["rv_60m", "tail", "volume_spike_60m"]
].mean()

,rv_60m,tail,volume_spike_60m
abs_skew,,,
"(-0.001, 6.66e-16]",0.000438,0.087918,0.042414
"(6.66e-16, 1.33e-15]",0.000399,0.073016,0.032054
"(1.33e-15, 2.846]",0.000540,0.155674,0.039688


In [ ]:
df["skew_bin"] = pd.qcut(
    df["skew_sentiment"],
    5,
    duplicates="drop"
)

df.groupby("skew_bin")[
    ["rv_60m","tail","volume_spike_60m"]
].mean()

,rv_60m,tail,volume_spike_60m
skew_bin,,,
"(-2.6679999999999997, -6.66e-16]",0.000476,0.107427,0.041118
"(-6.66e-16, 0.0]",0.000431,0.087031,0.042663
"(0.0, 2.846]",0.000507,0.142355,0.034752


In [ ]:
# Statistical disagreement vs LLM disagreement

In [ ]:
df[["event_disagreement", "day_disagreement"]].corr()

,event_disagreement,day_disagreement
event_disagreement,1.000000,0.044159
day_disagreement,0.044159,1.000000


In [ ]:
X_A = df[
    [
        "event_disagreement",
        "uncertainty",
        "skew_sentiment",
        "surprise"
    ]
]

X_A = sm.add_constant(X_A)
model_A = sm.Logit(df["tail"], X_A).fit()

Optimization terminated successfully.
         Current function value: 0.314925
         Iterations 7


In [ ]:
X_B = df[
    [
        "day_disagreement",
        "uncertainty",
        "skew_sentiment",
        "surprise"
    ]
]

X_B = sm.add_constant(X_B)
model_B = sm.Logit(df["tail"], X_B).fit()

Optimization terminated successfully.
         Current function value: 0.314779
         Iterations 7


In [ ]:
X_C = df[
    [
        "event_disagreement",
        "day_disagreement",
        "uncertainty",
        "skew_sentiment",
        "surprise"
    ]
]

X_C = sm.add_constant(X_C)
model_C = sm.Logit(df["tail"], X_C).fit()

Optimization terminated successfully.
         Current function value: 0.313365
         Iterations 7


In [ ]:
print(model_A.prsquared)
print(model_B.prsquared)
print(model_C.prsquared)

0.03155159921221329
0.03200100083645263
0.036348262754445915


In [ ]:
print(model_A.aic, model_A.bic)
print(model_B.aic, model_B.bic)
print(model_C.aic, model_C.bic)

6777.740230466547 6814.15120951193
6774.5997085249255 6811.010687570309
6746.220042090611 6789.91321694507


In [ ]:
# Likelihood Ratio Test
from statsmodels.stats.anova import anova_lm

lr = 2 * (model_C.llf - model_A.llf)
print("lr:", lr)

lr: 33.520188375935504


In [ ]:
print(
model_C.summary()
)

                           Logit Regression Results                           
Dep. Variable:                   tail   No. Observations:                10745
Model:                          Logit   Df Residuals:                    10739
Method:                           MLE   Df Model:                            5
Date:                Thu, 11 Jun 2026   Pseudo R-squ.:                 0.03635
Time:                        15:47:45   Log-Likelihood:                -3367.1
converged:                       True   LL-Null:                       -3494.1
Covariance Type:            nonrobust   LLR p-value:                 7.579e-53
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -3.3414      0.132    -25.319      0.000      -3.600      -3.083
event_disagreement     2.4774      0.446      5.550      0.000       1.602       3.352
day_disagreement    

In [ ]:
print(
np.exp(model_C.params)
)

const                  0.035389
event_disagreement    11.910828
day_disagreement       0.560002
uncertainty            5.523976
skew_sentiment         0.779337
surprise               0.443313
dtype: float64


In [ ]:
df["log_news"] = np.log1p(df["n_news"])

X = df[
[
"event_disagreement",
"day_disagreement",
"abs_skew",
"uncertainty",
"surprise",
"log_news"
]
]

X = sm.add_constant(X)

tail_model = sm.Logit(
    df["tail"],
    X
).fit()

print(tail_model.summary())

Optimization terminated successfully.
         Current function value: 0.305883
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                   tail   No. Observations:                10745
Model:                          Logit   Df Residuals:                    10738
Method:                           MLE   Df Model:                            6
Date:                Thu, 11 Jun 2026   Pseudo R-squ.:                 0.05936
Time:                        17:20:34   Log-Likelihood:                -3286.7
converged:                       True   LL-Null:                       -3494.1
Covariance Type:            nonrobust   LLR p-value:                 1.829e-86
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -5.3630      0.221    -24.246      0.000      -5.797      -4.929
event

In [ ]:
X = df[
[
"event_disagreement",
"day_disagreement",
"uncertainty",
"log_news"
]
]

X = sm.add_constant(X)

tail_model = sm.Logit(
    df["tail"],
    X
).fit()

print(tail_model.summary())

Optimization terminated successfully.
         Current function value: 0.305895
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                   tail   No. Observations:                10745
Model:                          Logit   Df Residuals:                    10740
Method:                           MLE   Df Model:                            4
Date:                Thu, 11 Jun 2026   Pseudo R-squ.:                 0.05932
Time:                        17:23:57   Log-Likelihood:                -3286.8
converged:                       True   LL-Null:                       -3494.1
Covariance Type:            nonrobust   LLR p-value:                 2.006e-88
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -5.3483      0.200    -26.773      0.000      -5.740      -4.957
event

In [ ]:
tail_model = sm.Logit(
    df["tail"],
    X
).fit(
    cov_type="HC3"
)

print(tail_model.summary())

Optimization terminated successfully.
         Current function value: 0.305895
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                   tail   No. Observations:                10745
Model:                          Logit   Df Residuals:                    10740
Method:                           MLE   Df Model:                            4
Date:                Thu, 11 Jun 2026   Pseudo R-squ.:                 0.05932
Time:                        17:25:17   Log-Likelihood:                -3286.8
converged:                       True   LL-Null:                       -3494.1
Covariance Type:                  HC3   LLR p-value:                 2.006e-88
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -5.3483      0.197    -27.135      0.000      -5.735      -4.962
event

In [ ]:
# Final model
X = df[
[
"event_disagreement",
"day_disagreement",
"uncertainty",
"log_news"
]
]

X = sm.add_constant(X)

tail_model = sm.Logit(
    df["tail"],
    X
).fit(cov_type="HC3")

Optimization terminated successfully.
         Current function value: 0.305895
         Iterations 7


In [ ]:
print(np.exp(tail_model.params))

const                 0.004756
event_disagreement    2.763025
day_disagreement      0.528370
uncertainty           6.269871
log_news              4.712262
dtype: float64
